> ### ⚠️ Select the **`Python 3 (croprow)`** kernel first
> Runs in the croprow env (Python **3.11**, numpy<2, OpenCV). If cell 1 throws `ModuleNotFoundError` (cv2 / yaml), the wrong interpreter is selected. In VSCode: **Select Kernel -> Select Another Kernel... -> Jupyter Kernel... -> Python 3 (croprow)**. Do not use the repo-root `.venv` / `.venv-1` (those lack the croprow deps by design).

# 08 - Package a portable, ready-to-train dataset (Option B)

Builds a **self-contained** YOLO folder the trainer can unzip and train on directly - no need to run 01, no machine-specific absolute paths in the image lists. Copies the images + writes mirrored box labels + a `data.yaml`, and drops in `set_yaml_path.py` and `README_TRAIN.md`.

The split matches 01 exactly (read from `data/train.txt` & `data/val.txt` when present). ~1.1 GB of images are copied.

In [1]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow/utils.py) so `import croprow.utils` works
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow import utils as U

# === CONFIG: the ONLY place to set the dataset location ===================
# Prefer the LETTUCE_ROOT env var. Otherwise edit LETTUCE_ROOT_DEFAULT below.
# (No absolute path is baked into utils.py -- it is supplied from here.)
LETTUCE_ROOT_DEFAULT = r"D:\croprow_dataset\LettuceMOTS"
LETTUCE_ROOT = os.environ.get("LETTUCE_ROOT", LETTUCE_ROOT_DEFAULT)
# ==========================================================================

root = U.resolve_lettuce_root(LETTUCE_ROOT)
DATA_DIR = REPO_ROOT / "croprow" / "data"
RESULTS_MD = REPO_ROOT / "croprow" / "RESULTS.md"
print("LETTUCE_ROOT :", root)
print("DATA_DIR     :", DATA_DIR)
import shutil

# ===================== CONFIG (edit here only) =====================
# Bundle output goes OUTSIDE the repo (it is large data). Default: next to the
# dataset. Change freely.
OUT_DIR = Path(os.environ.get(
    "CROPROW_BUNDLE",
    str(Path(LETTUCE_ROOT).parent / "croprow_lettuce_yolo")))
MAKE_ZIP = False   # set True to also produce OUT_DIR.zip (PNGs barely compress)
# ===================================================================
print("bundle out:", OUT_DIR)

LETTUCE_ROOT : D:\croprow_dataset\LettuceMOTS
DATA_DIR     : D:\sih26software\Crop_detection_management\croprow\data
bundle out: D:\croprow_dataset\croprow_lettuce_yolo


In [2]:
SET_PATH_PY = '"""Repoint data.yaml \'path:\' to THIS folder. Run once after unzipping:\n\n    python set_yaml_path.py\n\nUltralytics resolves a relative \'path:\' against its global datasets_dir, not\nthe yaml location, so an absolute \'path:\' matching where the bundle actually\nlives is the reliable, zero-surprise setup.\n"""\nfrom pathlib import Path\n\nhere = Path(__file__).resolve().parent\nyml = here / "data.yaml"\nlines = yml.read_text().splitlines()\nout = [f"path: {here.as_posix()}" if ln.startswith("path:") else ln for ln in lines]\nyml.write_text("\\n".join(out) + "\\n")\nprint("data.yaml \'path:\' set to", here)\n'
README_TRAIN = '# croprow lettuce - ready-to-train YOLO dataset (single class)\n\nSelf-contained detection dataset. One class: `lettuce`. Boxes derived from the\nLettuceMOTS segmentation polygons; split is **by sequence folder** (no frame\nleakage between train and val). Real data only.\n\n## Train\n1. (once, after unzipping) fix the dataset path:\n   ```\n   python set_yaml_path.py\n   ```\n2. install ultralytics + a torch build matching your GPU (see pytorch.org).\n3. train YOLO11n:\n   ```\n   yolo detect train data=data.yaml model=yolo11n.pt imgsz=640 epochs=100\n   ```\n   or open the repo\'s `croprow/notebooks/03_train.ipynb` and set\n   `DATA_YAML` to this folder\'s `data.yaml`.\n\n## Layout\n```\nimages/train/<seq>/*.png     labels/train/<seq>/*.txt   # "0 cx cy w h"\nimages/val/<seq>/*.png       labels/val/<seq>/*.txt\ndata.yaml                     # nc=1, names=[lettuce]\n```\nUltralytics finds each label by swapping `images` -> `labels` in the image path.\n'
print("bundle helper file contents loaded")

bundle helper file contents loaded


## 1. Use the SAME split as notebook 01

Derive train/val sequences from the committed list files so the bundle is identical to what 01 produced; fall back to the seeded split if they are absent.

In [3]:
def seqs_from_listfile(p):
    return sorted({Path(l).parent.name for l in Path(p).read_text().splitlines() if l.strip()})

tr_txt, va_txt = DATA_DIR / "train.txt", DATA_DIR / "val.txt"
if tr_txt.is_file() and va_txt.is_file():
    train_seqs, val_seqs = seqs_from_listfile(tr_txt), seqs_from_listfile(va_txt)
    print("split source: data/train.txt + val.txt (matches 01)")
else:
    train_seqs, val_seqs = U.split_sequences(U.labeled_sequences(root), val_frac=0.25, seed=42)
    print("split source: seeded split_sequences (01 lists not found)")
print("TRAIN seqs:", train_seqs)
print("VAL   seqs:", val_seqs)
assert not (set(train_seqs) & set(val_seqs)), "sequence leaked across split!"

split source: data/train.txt + val.txt (matches 01)
TRAIN seqs: ['0000', '0001', '0002', '0005', '0006', '0009', '0010']
VAL   seqs: ['0004', '0008']


## 2. Package (copy images + write mirrored box labels + data.yaml)

In [4]:
manifest = U.package_dataset(root, OUT_DIR, train_seqs, val_seqs, copy_images=True)
for split, info in manifest["splits"].items():
    print(f"{split:5s}: {info['images']:5d} images | {info['boxes']:6d} boxes | seqs={info['sequences']}")
print("data.yaml ->", manifest["data_yaml"])

train:   383 images |   5148 boxes | seqs=['0000', '0001', '0002', '0005', '0006', '0009', '0010']
val  :   213 images |   2696 boxes | seqs=['0004', '0008']
data.yaml -> D:\croprow_dataset\croprow_lettuce_yolo\data.yaml


## 3. Drop in the portability script + README, print the final yaml

In [5]:
(OUT_DIR / "set_yaml_path.py").write_text(SET_PATH_PY)
(OUT_DIR / "README_TRAIN.md").write_text(README_TRAIN)
print("wrote set_yaml_path.py and README_TRAIN.md\n")
print((OUT_DIR / "data.yaml").read_text())

wrote set_yaml_path.py and README_TRAIN.md

path: D:\croprow_dataset\croprow_lettuce_yolo
train: images/train
val: images/val
nc: 1
names:
- lettuce



## 4. Sanity-check the bundle (label lookup resolves) + optional zip

In [6]:
# YOLO derives label path by swapping images->labels; confirm it resolves.
sample_img = next((OUT_DIR / "images" / "train").rglob("*.png"))
sample_lab = Path(str(sample_img).replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}")).with_suffix(".txt")
print("sample image:", sample_img)
print("sample label:", sample_lab, "| exists:", sample_lab.is_file())

total_imgs = sum(1 for _ in (OUT_DIR / "images").rglob("*.png"))
total_labs = sum(1 for _ in (OUT_DIR / "labels").rglob("*.txt"))
print(f"bundle totals: {total_imgs} images, {total_labs} labels")
assert total_imgs == total_labs, "image/label count mismatch!"

if MAKE_ZIP:
    zpath = shutil.make_archive(str(OUT_DIR), "zip", root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name)
    print("zipped ->", zpath)
else:
    print("MAKE_ZIP is False -> folder bundle ready at", OUT_DIR)

sample image: D:\croprow_dataset\croprow_lettuce_yolo\images\train\0000\000000.png
sample label: D:\croprow_dataset\croprow_lettuce_yolo\labels\train\0000\000000.txt | exists: True
bundle totals: 596 images, 596 labels
MAKE_ZIP is False -> folder bundle ready at D:\croprow_dataset\croprow_lettuce_yolo
